# HorizonRev TRL PPO Training (Colab)

This notebook trains a tiny text policy on top of `HorizonRevEnv` using HF TRL PPO.
It also plots reward progression and compares random vs trained average reward.

In [ ]:
!pip install -q -r requirements.txt trl transformers torch accelerate

In [ ]:
import random
import numpy as np
import torch
import matplotlib.pyplot as plt

from transformers import AutoTokenizer
from trl import PPOConfig, PPOTrainer, AutoModelForCausalLMWithValueHead

from horizonrev import HorizonRevEnv, HorizonRevConfig
from horizonrev.dynamics.experiments import ACTION_NAMES, heuristic_action

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

In [ ]:
ACTIONS = list(range(8))
ACTION_TOKENS = [f'<A{i}>' for i in ACTIONS]

def obs_to_text(obs):
    return (
        f"month={obs[0]:.3f} arr={obs[1]:.3f} conv={obs[2]:.3f} churn={obs[3]:.3f} "
        f"discount={obs[4]:.3f} smb_dem={obs[5]:.3f} ent_dem={obs[6]:.3f} "
        f"pricing={int(obs[7])} onboarding={int(obs[8])} focus={int(obs[9])}. "
        f"Choose one action token from: {' '.join(ACTION_TOKENS)}"
    )

def decode_action(text):
    for i, tok in enumerate(ACTION_TOKENS):
        if tok in text:
            return i
    return random.randint(0, 7)

def evaluate(agent='random', episodes=20, seed_offset=1000, model=None, tokenizer=None):
    scores = []
    for ep in range(episodes):
        env = HorizonRevEnv(HorizonRevConfig.default())
        obs = env.reset(seed=seed_offset + ep)
        done = False
        total = 0.0
        while not done:
            if agent == 'random':
                action = random.randint(0, 7)
            elif agent == 'heuristic':
                action = heuristic_action(obs)
            else:
                prompt = obs_to_text(obs)
                q = tokenizer(prompt, return_tensors='pt').input_ids.to(model.pretrained_model.device)
                with torch.no_grad():
                    r = model.generate(q, max_new_tokens=4, do_sample=True, top_k=0, top_p=1.0)
                action_text = tokenizer.decode(r[0][q.shape[-1]:], skip_special_tokens=False)
                action = decode_action(action_text)
            obs, reward, done, _ = env.step(action)
            total += reward
        scores.append(total)
    return float(np.mean(scores)), scores

In [ ]:
model_name = 'sshleifer/tiny-gpt2'
tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.add_special_tokens({'additional_special_tokens': ACTION_TOKENS})

model = AutoModelForCausalLMWithValueHead.from_pretrained(model_name)
model.pretrained_model.resize_token_embeddings(len(tokenizer))
model.to(device)

ppo_config = PPOConfig(
    batch_size=4,
    mini_batch_size=2,
    learning_rate=1e-5,
    ppo_epochs=2,
    log_with=None,
)

ppo_trainer = PPOTrainer(
    config=ppo_config,
    model=model,
    ref_model=None,
    tokenizer=tokenizer,
)

baseline_avg, _ = evaluate('random', episodes=20)
print('Random baseline avg reward:', baseline_avg)

In [ ]:
episode_rewards = []

for ep in range(60):
    env = HorizonRevEnv(HorizonRevConfig.default())
    obs = env.reset(seed=ep)
    done = False
    total_reward = 0.0

    query_tensors = []
    response_tensors = []
    rewards = []

    while not done:
        prompt = obs_to_text(obs)
        q = tokenizer(prompt, return_tensors='pt').input_ids.to(model.pretrained_model.device)
        r = ppo_trainer.generate(q, max_new_tokens=4, do_sample=True, top_k=0, top_p=1.0)

        generated = r[0][q.shape[-1]:]
        action_text = tokenizer.decode(generated, skip_special_tokens=False)
        action = decode_action(action_text)

        obs, reward, done, _ = env.step(action)
        total_reward += reward

        query_tensors.append(q.squeeze(0))
        response_tensors.append(generated)
        rewards.append(torch.tensor(reward, dtype=torch.float32).to(model.pretrained_model.device))

    ppo_trainer.step(query_tensors, response_tensors, rewards)
    episode_rewards.append(total_reward)
    if (ep + 1) % 10 == 0:
        print(f'Episode {ep+1}: total_reward={total_reward:.3f}')

plt.figure(figsize=(7, 3.5))
plt.plot(episode_rewards)
plt.title('HorizonRev PPO Training Reward Curve')
plt.xlabel('Episode')
plt.ylabel('Total reward')
plt.show()

In [ ]:
trained_avg, trained_scores = evaluate('trained', episodes=20, model=model, tokenizer=tokenizer)
random_avg, random_scores = evaluate('random', episodes=20)

print(f'Random avg reward (20 eps):  {random_avg:.3f}')
print(f'Trained avg reward (20 eps): {trained_avg:.3f}')
print(f'Improvement: {trained_avg - random_avg:+.3f}')

torch.save(model.state_dict(), 'horizonrev_trl_policy.pt')
print('Saved model to horizonrev_trl_policy.pt')